# M0 feature engineering: past-only household demand, calendar and holiday features

This notebook consumes the Stage 3 household-day target CSV, adds leakage-safe predictors known by the start of each target date, and publishes CSV-only M0 datasets. P95 is the primary outcome and P90 is a sensitivity outcome. The same eligible household-date rows are used for both outcomes.

Historical measurement predictors apply the frozen feature-specific coverage policy from 03B, loaded from its versioned configuration. No weather is joined here. No features are imputed, no rows are split, and no model is fitted. Outer chronological split assignment belongs exclusively to `04_split_design.ipynb`.


## Stage 0 — Configuration and reproducibility

The target and feature-quality policies are read from their frozen upstream configurations; this notebook fails if either policy is not frozen. Run from the project root or the `notebooks` folder.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if not (PROJECT_ROOT / "data").is_dir() or not (PROJECT_ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Could not identify project root: {PROJECT_ROOT}")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
FEATURE_TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "04_feature_engineering"
MODEL_TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "05_model_ready"
for directory in (PROCESSED_DIR, FEATURE_TABLE_DIR, MODEL_TABLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "03_daily_with_target.csv"
INPUT_CONTRACT_PATH = PROCESSED_DIR / "03_daily_with_target_dtype_contract.json"
STAGE3_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "03_household_day_target" / "03_run_manifest.json"
TARGET_CONFIG_PATH = PROJECT_ROOT / "reports" / "tables" / "03a_target_design_experiments" / "03a_selected_target_config.json"
FEATURE_QUALITY_MANIFEST_PATH = PROJECT_ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments" / "03b_run_manifest.json"
FEATURE_QUALITY_CONFIG_PATH = PROJECT_ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments" / "03b_selected_feature_quality_config.json"
FEATURE_QUALITY_POLICY_PATH = PROJECT_ROOT / "reports" / "tables" / "03b_historical_feature_quality_experiments" / "03b_feature_quality_policy_candidates.csv"
HOLIDAY_PATH = INTERIM_DIR / "sri_lanka_holidays_2023_2024.csv"
FEATURE_OUTPUT_PATH = PROCESSED_DIR / "04_feature_dataset_m0.csv"
FEATURE_CONTRACT_PATH = PROCESSED_DIR / "04_feature_dataset_m0_dtype_contract.json"
CANONICAL_OUTPUT_PATH = PROCESSED_DIR / "05_model_ready_m0.csv"
CANONICAL_CONTRACT_PATH = PROCESSED_DIR / "05_model_ready_m0_dtype_contract.json"
P95_OUTPUT_PATH = PROCESSED_DIR / "05_model_ready_m0_p95.csv"
P95_CONTRACT_PATH = PROCESSED_DIR / "05_model_ready_m0_p95_dtype_contract.json"
P90_OUTPUT_PATH = PROCESSED_DIR / "05_model_ready_m0_p90.csv"
P90_CONTRACT_PATH = PROCESSED_DIR / "05_model_ready_m0_p90_dtype_contract.json"

CSV_ENCODING = "utf-8"
CSV_SEPARATOR = ","
CSV_MISSING_VALUE = ""
CSV_CHUNK_SIZE = 100_000
RANDOM_STATE = 42

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)

def file_sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()

def write_csv_atomic(frame, path):
    path = Path(path)
    temporary = path.with_name(path.stem + ".tmp" + path.suffix)
    if temporary.exists():
        temporary.unlink()
    frame.to_csv(temporary, index=False, encoding=CSV_ENCODING, sep=CSV_SEPARATOR,
                 na_rep=CSV_MISSING_VALUE, date_format="%Y-%m-%d %H:%M:%S", float_format="%.17g")
    temporary.replace(path)

def logical_dtype(dtype):
    if pd.api.types.is_datetime64_any_dtype(dtype): return "datetime"
    if pd.api.types.is_bool_dtype(dtype): return "boolean"
    if pd.api.types.is_integer_dtype(dtype): return str(dtype)
    if pd.api.types.is_float_dtype(dtype): return str(dtype)
    return "string"

def write_dtype_contract(frame, dataset_path, contract_path):
    columns = {name: logical_dtype(frame[name].dtype) for name in frame.columns}
    contract = {
        "contract_version": 1, "path": str(dataset_path), "format": "csv",
        "encoding": CSV_ENCODING, "separator": CSV_SEPARATOR, "missing_value": CSV_MISSING_VALUE,
        "date_columns": [name for name, dtype in columns.items() if dtype == "datetime"],
        "columns": columns,
        "reader": {"true_values": ["True"], "false_values": ["False"], "keep_default_na": False, "na_values": [""]},
    }
    write_json(contract, contract_path)
    return contract

def read_dataset_csv(path, contract, usecols=None, chunksize=None):
    selected = list(contract["columns"]) if usecols is None else list(usecols)
    date_columns = [name for name in contract.get("date_columns", []) if name in selected]
    dtype_map = {name: dtype for name, dtype in contract["columns"].items()
                 if name in selected and dtype != "datetime"}
    return pd.read_csv(path, usecols=selected, dtype=dtype_map or None,
                       parse_dates=date_columns or None,
                       true_values=contract["reader"]["true_values"],
                       false_values=contract["reader"]["false_values"],
                       keep_default_na=contract["reader"]["keep_default_na"],
                       na_values=contract["reader"]["na_values"],
                       encoding=contract["encoding"], sep=contract["separator"],
                       chunksize=chunksize, low_memory=False)

def save_table(frame, filename, title=None, n=25, model_ready=False):
    directory = MODEL_TABLE_DIR if model_ready else FEATURE_TABLE_DIR
    path = directory / filename
    if path.suffix.lower() != ".csv":
        raise ValueError(f"Report tables must be CSV: {path}")
    write_csv_atomic(frame, path)
    if title: print(f"{title}: {path.name}")
    display(frame.head(n))
    return path

required_inputs = [INPUT_PATH, INPUT_CONTRACT_PATH, STAGE3_MANIFEST_PATH, HOLIDAY_PATH, TARGET_CONFIG_PATH, FEATURE_QUALITY_MANIFEST_PATH, FEATURE_QUALITY_CONFIG_PATH, FEATURE_QUALITY_POLICY_PATH]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Required upstream artifacts are missing: {missing_inputs}")

stage3_manifest = read_json(STAGE3_MANIFEST_PATH)
input_contract = read_json(INPUT_CONTRACT_PATH)
target_config = read_json(TARGET_CONFIG_PATH)
feature_quality_manifest = read_json(FEATURE_QUALITY_MANIFEST_PATH)
feature_quality_config = read_json(FEATURE_QUALITY_CONFIG_PATH)
feature_quality_candidates = pd.read_csv(FEATURE_QUALITY_POLICY_PATH, low_memory=False)
target_policy_status = stage3_manifest.get("target_policy_status", "not_recorded")
if target_policy_status != "frozen_primary_research_policy" or target_config.get("status") != target_policy_status:
    raise RuntimeError("Stage 3 and 03A must both report the same frozen target policy before feature engineering.")
if stage3_manifest.get("target_configuration_id") != target_config.get("configuration_id"):
    raise RuntimeError("Stage 3 target configuration ID does not match the frozen 03A configuration.")
if feature_quality_manifest.get("upstream_target_configuration_id") != target_config.get("configuration_id"):
    raise RuntimeError("03B feature-quality evidence was generated under a different target configuration.")
if feature_quality_manifest.get("status") != "frozen_feature_quality_policy" or feature_quality_config.get("status") != "frozen_feature_quality_policy":
    raise RuntimeError("03B historical feature-quality policy is not frozen.")
if feature_quality_manifest.get("feature_quality_configuration_id") != feature_quality_config.get("configuration_id"):
    raise RuntimeError("03B run manifest and frozen feature-quality configuration IDs differ.")
if feature_quality_config.get("upstream_target_configuration_id") != target_config.get("configuration_id"):
    raise RuntimeError("03B frozen feature-quality policy was created under a different target policy.")
FEATURE_QUALITY_POLICY_ID = feature_quality_config["configuration_id"]
FEATURE_QUALITY_POLICY_STATUS = feature_quality_config["status"]
FEATURE_CANDIDATE_FEATURES = ["daily_peak_kw", "daily_mean_kw", "daily_std_kw", "import_active_ratio", "load_factor", "interval_p95_kw", "peak_time_minutes"]
if set(feature_quality_config.get("minimum_valid_intervals", {})) != set(FEATURE_CANDIDATE_FEATURES):
    raise RuntimeError("03B frozen feature-quality policy does not contain the exact expected feature set.")
feature_quality_cutoffs = {name: int(value) for name, value in feature_quality_config["minimum_valid_intervals"].items()}
if any(value not in [92, 93, 94, 95, 96] for value in feature_quality_cutoffs.values()):
    raise RuntimeError("03B frozen policy contains an unsupported minimum coverage.")
if not feature_quality_manifest.get("validation") or not all(feature_quality_manifest["validation"].values()):
    raise RuntimeError("03B feature-quality manifest contains a failed validation.")
if feature_quality_candidates.set_index("feature")["candidate_minimum_coverage"].astype(int).to_dict() != feature_quality_cutoffs:
    raise RuntimeError("Frozen 03B configuration differs from its empirical candidate evidence.")
target_signature = stage3_manifest["outputs"]["target_signature"]
if INPUT_PATH.stat().st_size != target_signature["size_bytes"]:
    raise RuntimeError("Stage 3 target CSV size differs from its run manifest")
if file_sha256(INPUT_PATH) != target_signature["sha256"]:
    raise RuntimeError("Stage 3 target CSV hash differs from its run manifest")

print(f"Project: {PROJECT_ROOT}")
print(f"Target-policy status: {target_policy_status} ({target_config['configuration_id']})")
print(f"Feature-quality policy: {FEATURE_QUALITY_POLICY_ID} [{FEATURE_QUALITY_POLICY_STATUS}]")
print(f"Feature-specific minimum intervals: {feature_quality_cutoffs}")
print(f"Stage 3 input: {INPUT_PATH} ({target_signature['rows']:,} rows)")
print(f"Holiday calendar: {HOLIDAY_PATH}")


## Stage 1 — Load the household-day labels and validated holiday calendar

The daily input is the Stage 3 all-calendar-day CSV. Holiday flags are joined using the prediction/target date. Days not listed as holidays receive `False`; the holiday name is retained only for audit context and is not a model predictor.


In [ ]:
SOURCE_COLUMNS = [
    "household_id", "measurement_date", "valid_interval_count",
    "observed_daily_peak_kw", "daily_peak_kw", "daily_mean_kw", "observed_daily_mean_kw",
    "observed_daily_std_kw", "daily_import_energy_kwh", "observed_import_active_ratio",
    "observed_load_factor", "observed_interval_p95_kw", "observed_peak_hour_sin", "observed_peak_hour_cos",
    "historical_peak_p95_45d", "historical_peak_p90_45d", "target_config_id",
    "target_eligible", "target_high_demand_p95", "target_high_demand_p90",
]
source_contract_missing = sorted(set(SOURCE_COLUMNS) - set(input_contract["columns"]))
if source_contract_missing:
    raise ValueError(f"Stage 3 dtype contract lacks required fields: {source_contract_missing}")

daily = read_dataset_csv(INPUT_PATH, input_contract, usecols=SOURCE_COLUMNS)
daily["household_id"] = daily["household_id"].astype("string")
daily["measurement_date"] = pd.to_datetime(daily["measurement_date"], errors="raise").dt.normalize()
daily = daily.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True)
if not daily["target_config_id"].astype(str).eq(target_config["configuration_id"]).all():
    raise RuntimeError("Stage 3 rows contain a target configuration ID inconsistent with the frozen 03A policy.")
daily["feature_quality_policy_id"] = FEATURE_QUALITY_POLICY_ID
daily["feature_quality_policy_status"] = FEATURE_QUALITY_POLICY_STATUS

quality_source_map = {
    "daily_peak_kw": "observed_daily_peak_kw",
    "daily_mean_kw": "observed_daily_mean_kw",
    "daily_std_kw": "observed_daily_std_kw",
    "import_active_ratio": "observed_import_active_ratio",
    "load_factor": "observed_load_factor",
    "interval_p95_kw": "observed_interval_p95_kw",
}
for policy_name, source_column in quality_source_map.items():
    daily[f"feature_{policy_name}"] = daily[source_column].where(daily["valid_interval_count"].ge(feature_quality_cutoffs[policy_name]))
peak_time_minimum = feature_quality_cutoffs["peak_time_minutes"]
daily["feature_peak_hour_sin"] = daily["observed_peak_hour_sin"].where(daily["valid_interval_count"].ge(peak_time_minimum))
daily["feature_peak_hour_cos"] = daily["observed_peak_hour_cos"].where(daily["valid_interval_count"].ge(peak_time_minimum))
if not daily["feature_daily_peak_kw"].equals(daily["daily_peak_kw"]):
    raise RuntimeError("03B historical peak candidate does not reproduce the Stage 3 qualifying peak series.")

daily["row_id"] = daily["household_id"].astype(str) + "|" + daily["measurement_date"].dt.strftime("%Y-%m-%d")

holidays = pd.read_csv(HOLIDAY_PATH, dtype={"date": "string", "holiday_name": "string", "source_reference": "string"},
                       true_values=["True", "true"], false_values=["False", "false"], keep_default_na=False)
required_holiday_columns = ["date", "holiday_name", "is_public_holiday", "is_poya_day", "source_reference"]
if holidays.columns.tolist() != required_holiday_columns:
    raise ValueError(f"Unexpected holiday schema: {holidays.columns.tolist()}")
holidays["date"] = pd.to_datetime(holidays["date"], format="%Y-%m-%d", errors="raise").dt.normalize()
for flag in ["is_public_holiday", "is_poya_day"]:
    if holidays[flag].dtype != bool:
        holidays[flag] = holidays[flag].astype("boolean")
if holidays["date"].duplicated().any() or holidays[required_holiday_columns].isna().any().any():
    raise ValueError("Holiday calendar dates must be unique and all required fields non-null")
if len(holidays) != 52:
    raise ValueError(f"Expected the validated 52-row holiday calendar; found {len(holidays)} rows")

data_years = set(daily["measurement_date"].dt.year.unique())
holiday_years = set(holidays["date"].dt.year.unique())
if not data_years.issubset(holiday_years):
    raise ValueError(f"Holiday calendar does not cover every data year: data={sorted(data_years)}, calendar={sorted(holiday_years)}")
holiday_join = holidays[["date", "holiday_name", "is_public_holiday", "is_poya_day"]].rename(columns={"date": "measurement_date"})
daily = daily.merge(holiday_join, on="measurement_date", how="left", validate="many_to_one", sort=False)
daily["is_public_holiday"] = daily["is_public_holiday"].fillna(False).astype(bool)
daily["is_poya_day"] = daily["is_poya_day"].fillna(False).astype(bool)
daily["holiday_name"] = daily["holiday_name"].astype("string")
daily = daily.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True)

input_checks = pd.DataFrame([
    ("row_count_matches_stage3", len(daily) == target_signature["rows"], f"{len(daily):,}"),
    ("household_count_matches_stage3", daily["household_id"].nunique() == stage3_manifest["population"]["households"], str(daily["household_id"].nunique())),
    ("household_date_key_unique", not daily.duplicated(["household_id", "measurement_date"]).any(), str(daily.duplicated(["household_id", "measurement_date"]).sum())),
    ("row_id_unique", daily["row_id"].is_unique, str(daily["row_id"].nunique())),
    ("frozen_target_config_matches_rows", daily["target_config_id"].astype(str).eq(target_config["configuration_id"]).all(), target_config["configuration_id"]),
    ("feature_quality_policy_matches_03b_manifest", feature_quality_manifest["upstream_target_configuration_id"] == target_config["configuration_id"], feature_quality_manifest["status"]),
    ("feature_specific_coverage_rules_applied", all(daily.loc[daily["valid_interval_count"].lt(feature_quality_cutoffs[name]), f"feature_{name}"].isna().all() for name in quality_source_map), str(feature_quality_cutoffs)),
    ("feature_peak_matches_frozen_history_rule", daily["feature_daily_peak_kw"].equals(daily["daily_peak_kw"]), f">={feature_quality_cutoffs['daily_peak_kw']}/96"),
    ("target_eligible_rows_match_manifest", int(daily["target_eligible"].sum()) == stage3_manifest["population"]["target_eligible_rows"], f"{int(daily['target_eligible'].sum()):,}"),
    ("eligible_p95_labels_binary", daily.loc[daily["target_eligible"], "target_high_demand_p95"].isin([0, 1]).all(), "eligible rows are 0/1"),
    ("eligible_p90_labels_binary", daily.loc[daily["target_eligible"], "target_high_demand_p90"].isin([0, 1]).all(), "eligible rows are 0/1"),
    ("same_population_has_both_targets", daily.loc[daily["target_eligible"], ["target_high_demand_p95", "target_high_demand_p90"]].notna().all().all(), "P95 and P90 labels present on the shared eligible set"),
    ("holiday_flags_non_null", daily[["is_public_holiday", "is_poya_day"]].notna().all().all(), "left-join misses set to false"),
    ("holiday_join_preserves_rows", len(daily) == target_signature["rows"], f"{len(daily):,}"),
], columns=["check", "passed", "evidence"])
save_table(input_checks, "04_00_input_and_holiday_checks.csv", "Input and holiday checks")
if not input_checks["passed"].all():
    raise RuntimeError(f"Input/holiday validation failed: {input_checks.loc[~input_checks.passed, 'check'].tolist()}")

display(daily[["measurement_date", "holiday_name", "is_public_holiday", "is_poya_day"]].drop_duplicates().query("is_public_holiday or is_poya_day").head(12))


## Stage 2 — Build past-only predictors

All consumption features use prior calendar dates only. Exact lags refer to exact dates because Stage 3 supplied the continuous household-day calendar; missing or below-policy values remain missing and are never replaced with another day. Peak, mean, active-import ratio, load factor, interval P95, and peak-time predictors each use the frozen minimum coverage supplied by 03B. Relative features use the Stage 3 past-only threshold.


In [ ]:
def engineer_household(group):
    g = group.sort_values("measurement_date", kind="mergesort").copy()
    date = g["measurement_date"]

    # Exact prior-calendar-day demand/quality lags.
    g["lag1_peak_kw"] = g["feature_daily_peak_kw"].shift(1)
    g["lag2_peak_kw"] = g["feature_daily_peak_kw"].shift(2)
    g["peak_d7"] = g["feature_daily_peak_kw"].shift(7)  # exact lag-7 and same weekday, stored once
    g["peak_d14"] = g["feature_daily_peak_kw"].shift(14)
    g["peak_d21"] = g["feature_daily_peak_kw"].shift(21)
    g["peak_d28"] = g["feature_daily_peak_kw"].shift(28)
    g["lag1_mean_kw"] = g["feature_daily_mean_kw"].shift(1)
    g["lag7_mean_kw"] = g["feature_daily_mean_kw"].shift(7)
    g["lag1_import_active_ratio"] = g["feature_import_active_ratio"].shift(1)
    g["lag7_import_active_ratio"] = g["feature_import_active_ratio"].shift(7)
    g["lag1_load_factor"] = g["feature_load_factor"].shift(1)
    g["lag1_interval_p95_kw"] = g["feature_interval_p95_kw"].shift(1)
    g["lag1_peak_hour_sin"] = g["feature_peak_hour_sin"].shift(1)
    g["lag1_peak_hour_cos"] = g["feature_peak_hour_cos"].shift(1)
    g["has_lag7"] = g["peak_d7"].notna()

    same_weekday_columns = ["peak_d7", "peak_d14", "peak_d21", "peak_d28"]
    g["same_weekday_valid_count_4"] = g[same_weekday_columns].notna().sum(axis=1).astype("int8")
    g["same_weekday_peak_mean4"] = g[same_weekday_columns].mean(axis=1)

    shifted_peak = g["feature_daily_peak_kw"].shift(1)
    for window in (3, 7, 30):
        rolling = shifted_peak.rolling(window=window, min_periods=1)
        g[f"peak_mean_{window}d"] = rolling.mean()
        g[f"valid_peak_days_{window}d"] = rolling.count().astype("int16")
        if window in (7, 30):
            g[f"peak_max_{window}d"] = rolling.max()
            g[f"peak_std_{window}d"] = rolling.std(ddof=0)
    g["peak_change_1d"] = g["lag1_peak_kw"] - g["lag2_peak_kw"]
    g["peak_mean3_over_30"] = g["peak_mean_3d"] / g["peak_mean_30d"].where(g["peak_mean_30d"].gt(0))
    g["peak_mean_7d_over_30d"] = g["peak_mean_7d"] / g["peak_mean_30d"].where(g["peak_mean_30d"].gt(0))

    for label, threshold_column in [("p95", "historical_peak_p95_45d"), ("p90", "historical_peak_p90_45d")]:
        threshold = g[threshold_column].where(g[threshold_column].gt(0))
        for lag, source in [(1, "lag1_peak_kw"), (2, "lag2_peak_kw"), (7, "peak_d7")]:
            g[f"lag{lag}_peak_over_{label}"] = g[source] / threshold
        g[f"lag1_peak_minus_{label}"] = g["lag1_peak_kw"] - g[threshold_column]
        g[f"peak_mean_7d_over_{label}"] = g["peak_mean_7d"] / threshold

    for label, target_column in [("p95", "target_high_demand_p95"), ("p90", "target_high_demand_p90")]:
        shifted_event = pd.to_numeric(g[target_column], errors="coerce").shift(1)
        for window in (7, 30):
            event_rolling = shifted_event.rolling(window=window, min_periods=1)
            g[f"{label}_high_count_{window}d"] = event_rolling.sum().fillna(0)
            g[f"{label}_eligible_event_days_{window}d"] = event_rolling.count().astype("int16")
        denominator = g[f"{label}_eligible_event_days_30d"].where(g[f"{label}_eligible_event_days_30d"].gt(0))
        g[f"{label}_high_rate_30d"] = g[f"{label}_high_count_30d"] / denominator
        prior_event_date = date.where(pd.to_numeric(g[target_column], errors="coerce").eq(1)).ffill().shift(1)
        g[f"{label}_days_since_last_high"] = (date - prior_event_date).dt.days.astype("float64")

    day_of_week = date.dt.dayofweek
    month = date.dt.month
    g["day_of_week_sin"] = np.sin(2 * np.pi * day_of_week / 7)
    g["day_of_week_cos"] = np.cos(2 * np.pi * day_of_week / 7)
    g["month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
    g["month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)
    g["is_weekend"] = day_of_week.ge(5).astype("int8")
    return g

parts = []
for _, household_group in daily.groupby("household_id", sort=False, observed=True):
    parts.append(engineer_household(household_group))
features = pd.concat(parts, ignore_index=True)
features = features.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True)
print(f"Feature construction complete: {len(features):,} household-day rows")


## Stage 3 — Feature registry, missingness, and redundancy audit

The registry states each feature's source, time shift, target-specific use, frozen 03B minimum historical coverage, and missing-value policy. Lagged energy is intentionally omitted as a predictor if it is effectively a deterministic 24-hour rescaling of daily mean power; the empirical relationship is saved for review. All predictor missing values remain missing for the later training-only preprocessing pipeline.


In [ ]:
COMMON_FEATURES = [
    "lag1_peak_kw", "lag2_peak_kw", "peak_d7", "peak_d14", "peak_d21", "peak_d28",
    "lag1_mean_kw", "lag7_mean_kw", "lag1_import_active_ratio", "lag7_import_active_ratio",
    "lag1_load_factor", "lag1_interval_p95_kw", "lag1_peak_hour_sin", "lag1_peak_hour_cos", "same_weekday_peak_mean4", "same_weekday_valid_count_4",
    "peak_mean_3d", "peak_mean_7d", "peak_mean_30d", "peak_max_7d", "peak_max_30d",
    "peak_std_7d", "peak_std_30d", "valid_peak_days_7d", "valid_peak_days_30d", "has_lag7",
    "peak_change_1d", "peak_mean3_over_30", "peak_mean_7d_over_30d",
    "day_of_week_sin", "day_of_week_cos", "month_sin", "month_cos", "is_weekend",
    "is_public_holiday", "is_poya_day",
]
P95_RELATIVE_FEATURES = ["lag1_peak_over_p95", "lag1_peak_minus_p95", "lag2_peak_over_p95", "lag7_peak_over_p95", "peak_mean_7d_over_p95"]
P90_RELATIVE_FEATURES = ["lag1_peak_over_p90", "lag1_peak_minus_p90", "lag2_peak_over_p90", "lag7_peak_over_p90", "peak_mean_7d_over_p90"]
P95_EVENT_FEATURES = ["p95_high_count_7d", "p95_high_count_30d", "p95_eligible_event_days_7d", "p95_eligible_event_days_30d", "p95_high_rate_30d", "p95_days_since_last_high"]
P90_EVENT_FEATURES = ["p90_high_count_7d", "p90_high_count_30d", "p90_eligible_event_days_7d", "p90_eligible_event_days_30d", "p90_high_rate_30d", "p90_days_since_last_high"]
P95_FEATURES = COMMON_FEATURES + P95_RELATIVE_FEATURES + P95_EVENT_FEATURES
P90_FEATURES = COMMON_FEATURES + P90_RELATIVE_FEATURES + P90_EVENT_FEATURES
ALL_FEATURES = COMMON_FEATURES + P95_RELATIVE_FEATURES + P90_RELATIVE_FEATURES + P95_EVENT_FEATURES + P90_EVENT_FEATURES

missing_features = sorted(set(ALL_FEATURES) - set(features.columns))
if missing_features:
    raise RuntimeError(f"Feature calculation omitted registered fields: {missing_features}")

def feature_metadata(name):
    if "peak_hour" in name:
        source, window, shift, group = "observed_peak_hour_sin/cos; prior day", 1, 1, "exact_calendar_lag"
    elif name in ["is_public_holiday", "is_poya_day", "is_weekend", "day_of_week_sin", "day_of_week_cos", "month_sin", "month_cos"]:
        source, window, shift, group = "measurement_date; holiday calendar" if "holiday" in name or "poya" in name else "measurement_date", 0, 0, "target_date_calendar"
    elif name.startswith("p95_") or name.startswith("p90_"):
        source = "target_high_demand_p95; target_eligible" if name.startswith("p95_") else "target_high_demand_p90; target_eligible"
        window = 30 if "30d" in name else (7 if "7d" in name else 0)
        shift, group = 1, "past_event_history"
    elif name.endswith("_over_p95") or name.endswith("_minus_p95"):
        source, window, shift, group = "lagged daily_peak_kw; historical_peak_p95_45d", 45, (int(re.match(r"^lag(\d+)_", name).group(1)) if re.match(r"^lag(\d+)_", name) else 1), "past_relative_demand"
    elif name.endswith("_over_p90") or name.endswith("_minus_p90"):
        source, window, shift, group = "lagged daily_peak_kw; historical_peak_p90_45d", 45, (int(re.match(r"^lag(\d+)_", name).group(1)) if re.match(r"^lag(\d+)_", name) else 1), "past_relative_demand"
    elif name in ["is_weekend"] or name.endswith("_sin") or name.endswith("_cos"):
        source, window, shift, group = "measurement_date", 0, 0, "target_date_calendar"
    elif name.startswith("lag") or name.startswith("peak_d"):
        source = "daily_peak_kw" if "peak" in name or name.startswith("peak_d") else ("feature_daily_mean_kw" if "mean" in name else ("feature_import_active_ratio" if "active_ratio" in name else ("feature_load_factor" if "load_factor" in name else "feature_interval_p95_kw")))
        if name.startswith("peak_d"):
            match = re.fullmatch(r"peak_d(\d+)", name)
            if match is None: raise RuntimeError(f"Malformed exact peak lag feature: {name}")
            window = shift = int(match.group(1))
        else:
            match = re.match(r"^lag(\d+)_", name)
            if match is None: raise RuntimeError(f"Malformed prefixed lag feature: {name}")
            window = shift = int(match.group(1))
        group = "exact_calendar_lag"
    elif name.startswith("same_weekday") or name in ["same_weekday_peak_mean4", "same_weekday_valid_count_4"]:
        source, window, shift, group = "daily_peak_kw", 28, 1, "same_weekday_history"
    elif name.startswith("valid_peak_days") or name == "has_lag7":
        source, window, shift, group = "daily_peak_kw", (30 if "30d" in name else 7), 1, "availability_context"
    elif name.startswith("peak_"):
        source, window, shift, group = "daily_peak_kw", (30 if "30" in name else (7 if "7" in name else 3)), 1, "past_rolling_or_trend"
    else:
        source, window, shift, group = "daily household-day fields", 0, 1, "past_only"
    target = "both" if name in COMMON_FEATURES else ("P95" if name in P95_RELATIVE_FEATURES + P95_EVENT_FEATURES else "P90")
    return (name, group, source, window, shift, target, "past-only or target-date-known",
            "retain missing; training-only imputation with missingness indicators")

feature_registry = pd.DataFrame(
    [feature_metadata(name) for name in ALL_FEATURES],
    columns=["feature_name", "feature_group", "source_columns", "window_days", "shift_days", "target_scope", "leakage_status", "missingness_policy"],
)
feature_registry["weather_feature"] = False
feature_registry["primary_predictor"] = True
def registry_lag_days(name):
    match = re.match(r"^lag(\d+)_", name)
    if match: return int(match.group(1))
    match = re.fullmatch(r"peak_d(\d+)", name)
    if match: return int(match.group(1))
    return pd.NA

def registry_rolling_window_days(name):
    if name.endswith("_over_p95") or name.endswith("_minus_p95") or name.endswith("_over_p90") or name.endswith("_minus_p90"):
        return 45
    match = re.search(r"_(\d+)d(?:_|$)", name)
    if match: return int(match.group(1))
    if name.startswith("same_weekday_"): return 28
    return pd.NA

feature_registry["lag_days"] = [registry_lag_days(name) for name in feature_registry["feature_name"]]
feature_registry["rolling_window_days"] = [registry_rolling_window_days(name) for name in feature_registry["feature_name"]]

def feature_quality_rule(name):
    if "peak_hour" in name:
        return feature_quality_cutoffs["peak_time_minutes"]
    if "active_ratio" in name:
        return feature_quality_cutoffs["import_active_ratio"]
    if "load_factor" in name:
        return feature_quality_cutoffs["load_factor"]
    if "interval_p95" in name:
        return feature_quality_cutoffs["interval_p95_kw"]
    if "mean_kw" in name:
        return feature_quality_cutoffs["daily_mean_kw"]
    if "peak" in name or name.startswith("valid_peak_days") or name == "has_lag7":
        return feature_quality_cutoffs["daily_peak_kw"]
    return pd.NA

feature_registry["source_min_valid_intervals"] = [feature_quality_rule(name) for name in feature_registry["feature_name"]]
feature_registry["feature_quality_policy_id"] = FEATURE_QUALITY_POLICY_ID
feature_registry["feature_quality_policy_status"] = FEATURE_QUALITY_POLICY_STATUS

def explicit_lag_metadata_valid(row):
    name = row.feature_name
    if name.startswith("lag"):
        match = re.match(r"^lag(\d+)_", name)
        return bool(match and int(row.lag_days) == int(match.group(1)))
    if name.startswith("peak_d"):
        match = re.fullmatch(r"peak_d(\d+)", name)
        return bool(match and int(row.lag_days) == int(match.group(1)))
    return True

lag_registry_valid = feature_registry.apply(explicit_lag_metadata_valid, axis=1)
if not lag_registry_valid.all():
    raise RuntimeError(f"Feature registry contains invalid exact-lag metadata: {feature_registry.loc[~lag_registry_valid, ['feature_name','lag_days','rolling_window_days','window_days','shift_days']].to_dict('records')}")
if int(feature_registry.loc[feature_registry.feature_name.eq("lag1_interval_p95_kw"), "lag_days"].iloc[0]) != 1:
    raise RuntimeError("lag1_interval_p95_kw must have a one-day lag; embedded p95 must not affect lag metadata.")
save_table(feature_registry, "04_01_feature_registry.csv", "M0 feature registry", n=80)

feature_missingness = pd.DataFrame({
    "feature_name": ALL_FEATURES,
    "missing_rows": [int(features[name].isna().sum()) for name in ALL_FEATURES],
})
feature_missingness["missing_rate"] = feature_missingness["missing_rows"] / len(features)
feature_missingness = feature_missingness.sort_values("missing_rate", ascending=False).reset_index(drop=True)
save_table(feature_missingness, "04_02_feature_missingness.csv", "Feature missingness", n=80)

energy_pair = daily[["daily_mean_kw", "daily_import_energy_kwh"]].dropna()
energy_correlation = float(energy_pair.corr().iloc[0, 1]) if len(energy_pair) > 1 else np.nan
energy_per_mean = float((energy_pair["daily_import_energy_kwh"] / energy_pair["daily_mean_kw"].where(energy_pair["daily_mean_kw"].ne(0))).median()) if len(energy_pair) else np.nan
energy_redundant = bool(np.isfinite(energy_correlation) and abs(energy_correlation) >= 0.999999 and np.isclose(energy_per_mean, 24.0, rtol=0.01, atol=0.01))
energy_redundancy = pd.DataFrame([{
    "paired_complete_days": len(energy_pair), "daily_mean_energy_correlation": energy_correlation,
    "median_energy_to_mean_ratio": energy_per_mean, "energy_is_24h_rescaling": energy_redundant,
    "feature_decision": "omit lagged energy predictors; preserve mean-power lags" if energy_redundant else "omit lagged energy predictors to avoid unnecessary redundancy; revisit if required",
}])
save_table(energy_redundancy, "04_03_energy_redundancy_audit.csv", "Mean power / energy redundancy audit")


## Stage 4 — Leakage, target separation, and exact feature validation

Checks include manual recomputation of exact/calendar-window examples and a future perturbation test. The future perturbation changes a later daily peak and verifies that already earlier predictors are unchanged. Same-day demand, labels, identifiers, and audit fields are not predictor columns.


In [ ]:
PREDICTOR_COLUMNS = sorted(set(P95_FEATURES + P90_FEATURES))
FORBIDDEN_PREDICTOR_COLUMNS = {
    "daily_peak_kw", "daily_mean_kw", "daily_import_energy_kwh", "target_high_demand_p95", "target_high_demand_p90",
    "target_eligible", "historical_peak_p95_45d", "historical_peak_p90_45d", "target_day_peak_kw",
    "household_id", "row_id", "measurement_date", "split_name", "valid_interval_count", "target_config_id",
    "feature_quality_policy_id", "feature_quality_policy_status", "feature_daily_peak_kw", "feature_daily_mean_kw",
    "feature_daily_std_kw", "feature_import_active_ratio", "feature_load_factor", "feature_interval_p95_kw",
    "feature_peak_hour_sin", "feature_peak_hour_cos",
}

def scalar_close(left, right):
    left_value = pd.to_numeric(pd.Series([left]), errors="coerce").to_numpy(dtype="float64", na_value=np.nan)[0]
    right_value = pd.to_numeric(pd.Series([right]), errors="coerce").to_numpy(dtype="float64", na_value=np.nan)[0]
    return bool(np.isclose(left_value, right_value, equal_nan=True))

sample_errors = []
eligible_sample = features.loc[features["target_eligible"]].sample(n=min(250, int(features["target_eligible"].sum())), random_state=RANDOM_STATE)
group_map = {household_id: group.set_index("measurement_date") for household_id, group in features.groupby("household_id", sort=False, observed=True)}
for row in eligible_sample.itertuples(index=False):
    group = group_map[row.household_id]
    date = row.measurement_date
    expected_lag1 = group.loc[date - pd.Timedelta(days=1), "daily_peak_kw"]
    expected_peak_d7 = group.loc[date - pd.Timedelta(days=7), "daily_peak_kw"]
    expected_history = group.loc[(group.index >= date - pd.Timedelta(days=7)) & (group.index < date), "daily_peak_kw"].dropna()
    expected_mean7 = group.loc[(group.index >= date - pd.Timedelta(days=7)) & (group.index < date), "feature_daily_peak_kw"].dropna().mean() if len(expected_history) else np.nan
    expected_lag1_mean = group.loc[date - pd.Timedelta(days=1), "feature_daily_mean_kw"]
    expected_lag1_active = group.loc[date - pd.Timedelta(days=1), "feature_import_active_ratio"]
    expected_lag1_peak_time_sin = group.loc[date - pd.Timedelta(days=1), "feature_peak_hour_sin"]
    if (not scalar_close(row.lag1_peak_kw, expected_lag1)
        or not scalar_close(row.peak_d7, expected_peak_d7)
        or not scalar_close(row.peak_mean_7d, expected_mean7)
        or not scalar_close(row.lag1_mean_kw, expected_lag1_mean)
        or not scalar_close(row.lag1_import_active_ratio, expected_lag1_active)
        or not scalar_close(row.lag1_peak_hour_sin, expected_lag1_peak_time_sin)):
        sample_errors.append(row.row_id)

test_household_id = features.groupby("household_id", observed=True).size().idxmax()
test_group = features.loc[features["household_id"].eq(test_household_id)].copy()
test_group_features = engineer_household(test_group)
perturb_position = int(len(test_group) * 3 // 4)
perturb_date = test_group.iloc[perturb_position]["measurement_date"]
perturbed_group = test_group.copy()
perturbed_group.loc[perturbed_group.index[perturb_position], "feature_daily_peak_kw"] = 999_999.0
perturbed_features = engineer_household(perturbed_group)
past_positions = test_group_features["measurement_date"].lt(perturb_date)
future_perturbation_unchanged = test_group_features.loc[past_positions, PREDICTOR_COLUMNS].reset_index(drop=True).equals(
    perturbed_features.loc[past_positions, PREDICTOR_COLUMNS].reset_index(drop=True)
)

leakage_checks = pd.DataFrame([
    ("all_household_days_preserved", len(features) == len(daily) == target_signature["rows"], f"{len(features):,}"),
    ("unique_household_date_and_row_id", not features.duplicated(["household_id", "measurement_date"]).any() and features["row_id"].is_unique, str(features["row_id"].nunique())),
    ("manual_lag_and_rolling_recomputation", len(sample_errors) == 0, str(sample_errors[:10])),
    ("future_perturbation_leaves_past_unchanged", future_perturbation_unchanged, f"household={test_household_id}; altered_date={perturb_date.date()}"),
    ("no_forbidden_predictors", not FORBIDDEN_PREDICTOR_COLUMNS.intersection(PREDICTOR_COLUMNS), str(sorted(FORBIDDEN_PREDICTOR_COLUMNS.intersection(PREDICTOR_COLUMNS)))),
    ("p95_excludes_p90_relative_and_event_features", not set(P90_RELATIVE_FEATURES + P90_EVENT_FEATURES).intersection(P95_FEATURES), "separate P95-specific threshold and event history"),
    ("p90_excludes_p95_relative_and_event_features", not set(P95_RELATIVE_FEATURES + P95_EVENT_FEATURES).intersection(P90_FEATURES), "separate P90-specific threshold and event history"),
    ("no_weather_predictors", not any(token in name.lower() for name in PREDICTOR_COLUMNS for token in ["weather", "temperature", "humidity", "precip"]), "M0 without weather"),
    ("no_infinite_predictors", not np.isinf(features[PREDICTOR_COLUMNS].select_dtypes(include=[np.number]).astype("float64").to_numpy()).any(), "finite or missing only"),
    ("no_predictor_imputation", True, "missing values remain for training-only preprocessing"),
    ("below_threshold_history_features_are_missing", all(features.loc[features["valid_interval_count"].lt(feature_quality_cutoffs[name]), f"feature_{name}"].isna().all() for name in quality_source_map), str(feature_quality_cutoffs)),
    ("candidate_peak_time_features_are_missing_below_support", features.loc[features["valid_interval_count"].lt(feature_quality_cutoffs['peak_time_minutes']), ["feature_peak_hour_sin", "feature_peak_hour_cos"]].isna().all().all(), f">={feature_quality_cutoffs['peak_time_minutes']}/96"),
    ("frozen_feature_quality_policy_applied", daily["feature_quality_policy_id"].eq(FEATURE_QUALITY_POLICY_ID).all() and daily["feature_quality_policy_status"].eq("frozen_feature_quality_policy").all(), FEATURE_QUALITY_POLICY_ID),
    ("exact_lag_registry_metadata_valid", bool(lag_registry_valid.all()), "lag prefixes parsed independently from p90/p95 feature names"),
], columns=["check", "passed", "evidence"])
save_table(leakage_checks, "04_04_leakage_and_feature_checks.csv", "Leakage and feature checks", n=30)
if not leakage_checks["passed"].all():
    raise RuntimeError(f"Leakage/features validation failed: {leakage_checks.loc[~leakage_checks.passed, 'check'].tolist()}")


## Stage 5 — Save the all-day M0 feature table as CSV

This full-calendar artifact preserves the daily key and labels for audit, together with the registered predictor columns and the 45-day target thresholds used to form labels. It is not itself a training matrix. CSV dtype contracts preserve dates, booleans, strings, and missing predictor values.


In [ ]:
FEATURE_AUDIT_COLUMNS = [
    "historical_peak_p95_45d", "historical_peak_p90_45d", "target_eligible",
    "target_high_demand_p95", "target_high_demand_p90", "holiday_name",
    "target_config_id", "feature_quality_policy_id", "feature_quality_policy_status",
]
feature_output_columns = ["row_id", "household_id", "measurement_date", *FEATURE_AUDIT_COLUMNS, *ALL_FEATURES]
feature_output = features[feature_output_columns].copy()
write_csv_atomic(feature_output, FEATURE_OUTPUT_PATH)
feature_contract = write_dtype_contract(feature_output, FEATURE_OUTPUT_PATH, FEATURE_CONTRACT_PATH)

feature_roundtrip_rows = 0
roundtrip_missing = pd.Series(0, index=ALL_FEATURES, dtype="int64")
sample_ids = set(feature_output.sample(n=min(500, len(feature_output)), random_state=RANDOM_STATE)["row_id"].astype(str))
roundtrip_parts = []
for chunk in read_dataset_csv(FEATURE_OUTPUT_PATH, feature_contract, chunksize=CSV_CHUNK_SIZE):
    feature_roundtrip_rows += len(chunk)
    roundtrip_missing = roundtrip_missing.add(chunk[ALL_FEATURES].isna().sum(), fill_value=0).astype("int64")
    sample_rows = chunk.loc[chunk["row_id"].astype(str).isin(sample_ids), ["row_id", *ALL_FEATURES]]
    if not sample_rows.empty:
        roundtrip_parts.append(sample_rows)
feature_sample_roundtrip = pd.concat(roundtrip_parts, ignore_index=True).sort_values("row_id").reset_index(drop=True)
feature_sample_source = feature_output.loc[feature_output["row_id"].astype(str).isin(sample_ids), ["row_id", *ALL_FEATURES]].sort_values("row_id").reset_index(drop=True)
feature_sample_values_match = feature_sample_roundtrip["row_id"].astype(str).equals(feature_sample_source["row_id"].astype(str))
if feature_sample_values_match:
    for column in ALL_FEATURES:
        actual = pd.to_numeric(feature_sample_roundtrip[column], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        expected = pd.to_numeric(feature_sample_source[column], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        if not np.allclose(actual, expected, equal_nan=True, rtol=1e-12, atol=1e-12):
            feature_sample_values_match = False
            break

feature_csv_checks = pd.DataFrame([
    ("feature_csv_rows", feature_roundtrip_rows == len(feature_output) == target_signature["rows"], f"{feature_roundtrip_rows:,}"),
    ("feature_csv_missingness_preserved", roundtrip_missing.equals(feature_output[ALL_FEATURES].isna().sum().astype("int64")), "all predictor missing-value counts match"),
    ("sampled_feature_values_preserved", feature_sample_values_match, f"sample rows={len(feature_sample_roundtrip):,}"),
    ("feature_contract_exists", FEATURE_CONTRACT_PATH.exists(), str(FEATURE_CONTRACT_PATH)),
    ("feature_output_is_csv", FEATURE_OUTPUT_PATH.suffix.lower() == ".csv", FEATURE_OUTPUT_PATH.name),
], columns=["check", "passed", "evidence"])
save_table(feature_csv_checks, "04_05_feature_csv_validation.csv", "Feature CSV validation")
if not feature_csv_checks["passed"].all():
    raise RuntimeError(f"Feature CSV validation failed: {feature_csv_checks.loc[~feature_csv_checks.passed, 'check'].tolist()}")
feature_signature = {"path": str(FEATURE_OUTPUT_PATH), "format": "csv", "size_bytes": FEATURE_OUTPUT_PATH.stat().st_size,
                    "rows": len(feature_output), "columns": len(feature_output.columns), "sha256": file_sha256(FEATURE_OUTPUT_PATH)}
print(f"All-day feature table saved: {FEATURE_OUTPUT_PATH} ({len(feature_output):,} rows)")


## Stage 6 — Create canonical, P95, and P90 M0 model-ready CSVs

Only target-eligible rows with observed binary labels enter supervised datasets. P95 and P90 exports use the same ordered household-date rows, but each has its own historical-threshold-relative and event-history predictors. No outer split is assigned here; `04_split_design.ipynb` will create and freeze it once for both targets.


In [ ]:
eligible = feature_output.loc[feature_output["target_eligible"]].copy().reset_index(drop=True)
if eligible[["target_high_demand_p95", "target_high_demand_p90"]].isna().any().any():
    raise RuntimeError("Eligible rows must have both P95 and P90 labels")

MODEL_KEY_COLUMNS = ["row_id", "household_id", "measurement_date"]
canonical_columns = MODEL_KEY_COLUMNS + COMMON_FEATURES + P95_RELATIVE_FEATURES + P90_RELATIVE_FEATURES + P95_EVENT_FEATURES + P90_EVENT_FEATURES + ["target_high_demand_p95", "target_high_demand_p90"]
canonical = eligible[canonical_columns].copy()
p95_dataset = eligible[MODEL_KEY_COLUMNS + P95_FEATURES].copy()
p95_dataset["target"] = eligible["target_high_demand_p95"].astype("int8")
p90_dataset = eligible[MODEL_KEY_COLUMNS + P90_FEATURES].copy()
p90_dataset["target"] = eligible["target_high_demand_p90"].astype("int8")

for frame, dataset_path, contract_path in [
    (canonical, CANONICAL_OUTPUT_PATH, CANONICAL_CONTRACT_PATH),
    (p95_dataset, P95_OUTPUT_PATH, P95_CONTRACT_PATH),
    (p90_dataset, P90_OUTPUT_PATH, P90_CONTRACT_PATH),
]:
    write_csv_atomic(frame, dataset_path)
    write_dtype_contract(frame, dataset_path, contract_path)

split_prevalence = pd.DataFrame([
    {"target": "P95_primary", "eligible_rows": len(p95_dataset), "positive_events": int(p95_dataset["target"].sum()), "positive_rate": float(p95_dataset["target"].mean())},
    {"target": "P90_sensitivity", "eligible_rows": len(p90_dataset), "positive_events": int(p90_dataset["target"].sum()), "positive_rate": float(p90_dataset["target"].mean())},
])
save_table(split_prevalence, "05_00_target_class_distribution.csv", "Unsplit target class distribution", model_ready=True)

dataset_checks = pd.DataFrame([
    ("model_rows_equal_target_eligible", len(eligible) == stage3_manifest["population"]["target_eligible_rows"], f"{len(eligible):,}"),
    ("canonical_p95_p90_rows_match", len(canonical) == len(p95_dataset) == len(p90_dataset), f"{len(canonical):,}"),
    ("canonical_row_id_unique", canonical["row_id"].is_unique, str(canonical["row_id"].nunique())),
    ("same_ordered_row_ids", canonical["row_id"].equals(p95_dataset["row_id"]) and canonical["row_id"].equals(p90_dataset["row_id"]), "P95 and P90 populations identical"),
    ("p95_events_match_manifest", int(p95_dataset["target"].sum()) == stage3_manifest["target_results"]["p95_positive_events"], str(int(p95_dataset["target"].sum()))),
    ("p90_events_match_manifest", int(p90_dataset["target"].sum()) == stage3_manifest["target_results"]["p90_positive_events"], str(int(p90_dataset["target"].sum()))),
    ("no_date_split_created", "split_name" not in canonical.columns, "outer split reserved for 04_split_design.ipynb"),
    ("p95_has_only_p95_target_history", not set(P90_RELATIVE_FEATURES + P90_EVENT_FEATURES).intersection(p95_dataset.columns), "P95-specific feature set"),
    ("p90_has_only_p90_target_history", not set(P95_RELATIVE_FEATURES + P95_EVENT_FEATURES).intersection(p90_dataset.columns), "P90-specific feature set"),
    ("targets_are_binary", p95_dataset["target"].isin([0, 1]).all() and p90_dataset["target"].isin([0, 1]).all(), "both labels in {0,1}"),
    ("model_predictors_numeric_or_boolean", all(pd.api.types.is_numeric_dtype(frame[name].dtype) or pd.api.types.is_bool_dtype(frame[name].dtype) for frame, names in [(p95_dataset, P95_FEATURES), (p90_dataset, P90_FEATURES)] for name in names), "IDs and dates are metadata; predictors are numeric/boolean"),
    ("missing_predictors_retained", p95_dataset[P95_FEATURES].isna().any().any() and p90_dataset[P90_FEATURES].isna().any().any(), "no global imputation"),
    ("model_outputs_are_csv", all(path.suffix.lower() == ".csv" for path in [CANONICAL_OUTPUT_PATH, P95_OUTPUT_PATH, P90_OUTPUT_PATH]), "CSV only"),
], columns=["check", "passed", "evidence"])
save_table(dataset_checks, "05_01_model_dataset_checks.csv", "Model dataset checks", n=40, model_ready=True)
if not dataset_checks["passed"].all():
    raise RuntimeError(f"Model dataset validation failed: {dataset_checks.loc[~dataset_checks.passed, 'check'].tolist()}")



In [ ]:
def validate_model_csv(dataset_path, contract_path, expected, feature_columns):
    contract = read_json(contract_path)
    reloaded = read_dataset_csv(dataset_path, contract)
    checks = {
        "rows": len(reloaded) == len(expected),
        "row_id_order": reloaded["row_id"].astype(str).equals(expected["row_id"].astype(str)),
        "missingness": reloaded[feature_columns].isna().sum().equals(expected[feature_columns].isna().sum()),
        "feature_values": all(np.allclose(pd.to_numeric(reloaded[column], errors="coerce").to_numpy(dtype="float64", na_value=np.nan),
                                           pd.to_numeric(expected[column], errors="coerce").to_numpy(dtype="float64", na_value=np.nan),
                                           equal_nan=True, rtol=1e-12, atol=1e-12) for column in feature_columns),
        "targets": reloaded["target"].equals(expected["target"]),
    }
    return checks

p95_roundtrip = validate_model_csv(P95_OUTPUT_PATH, P95_CONTRACT_PATH, p95_dataset, P95_FEATURES)
p90_roundtrip = validate_model_csv(P90_OUTPUT_PATH, P90_CONTRACT_PATH, p90_dataset, P90_FEATURES)
model_csv_roundtrip = pd.DataFrame([
    {"dataset": "P95", "check": check, "passed": passed} for check, passed in p95_roundtrip.items()
] + [
    {"dataset": "P90", "check": check, "passed": passed} for check, passed in p90_roundtrip.items()
])
save_table(model_csv_roundtrip, "05_02_model_csv_roundtrip.csv", "Model CSV round-trip", n=20, model_ready=True)
if not model_csv_roundtrip["passed"].all():
    raise RuntimeError("One or more model-ready CSV round-trip checks failed")



## Stage 7 — Final validation and run manifest

The manifest links the feature outputs to the exact Stage 3 target CSV and holiday calendar, records P95/P90 predictor lists, reports class counts, and explicitly states that split assignment is deferred. A passing final validation is required before downstream EDA/split design.


In [ ]:
all_checks = pd.concat([
    input_checks.assign(stage="input"), leakage_checks.assign(stage="leakage"),
    feature_csv_checks.assign(stage="feature_csv"), dataset_checks.assign(stage="model_datasets"),
    model_csv_roundtrip.assign(stage="model_csv_roundtrip").rename(columns={"dataset": "check_group"}),
], ignore_index=True, sort=False)
save_table(all_checks, "04_06_final_validation.csv", "Feature engineering final validation", n=100)
if not all_checks["passed"].fillna(False).all():
    raise RuntimeError(f"Final feature-engineering validation failed: {all_checks.loc[~all_checks['passed'].fillna(False), ['stage', 'check']].to_dict('records')}")

canonical_signature = {"path": str(CANONICAL_OUTPUT_PATH), "format": "csv", "size_bytes": CANONICAL_OUTPUT_PATH.stat().st_size,
                       "rows": len(canonical), "columns": len(canonical.columns), "sha256": file_sha256(CANONICAL_OUTPUT_PATH)}
p95_signature = {"path": str(P95_OUTPUT_PATH), "format": "csv", "size_bytes": P95_OUTPUT_PATH.stat().st_size,
                 "rows": len(p95_dataset), "columns": len(p95_dataset.columns), "sha256": file_sha256(P95_OUTPUT_PATH)}
p90_signature = {"path": str(P90_OUTPUT_PATH), "format": "csv", "size_bytes": P90_OUTPUT_PATH.stat().st_size,
                 "rows": len(p90_dataset), "columns": len(p90_dataset.columns), "sha256": file_sha256(P90_OUTPUT_PATH)}

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "stage": "M0_feature_engineering",
    "feature_set": "M0_consumption_calendar_holiday_no_weather", "weather_included": False,
    "target_policy_status": target_policy_status, "target_policy": stage3_manifest["configuration"],
    "target_configuration_id": target_config["configuration_id"],
    "feature_quality_policy": {
        "policy_id": FEATURE_QUALITY_POLICY_ID,
        "status": FEATURE_QUALITY_POLICY_STATUS,
        "configuration_id": FEATURE_QUALITY_POLICY_ID,
        "configuration_path": str(FEATURE_QUALITY_CONFIG_PATH),
        "configuration_sha256": file_sha256(FEATURE_QUALITY_CONFIG_PATH),
        "source_manifest": str(FEATURE_QUALITY_MANIFEST_PATH),
        "source_candidate_table": str(FEATURE_QUALITY_POLICY_PATH),
        "minimum_valid_intervals": feature_quality_cutoffs,
        "daily_energy_policy": feature_quality_config["daily_energy_policy"],
        "selection_basis": feature_quality_config["decision_basis"],
        "researcher_review_required": False,
    },
    "input": {"path": str(INPUT_PATH), "sha256": target_signature["sha256"], "rows": len(daily), "columns": len(input_contract["columns"])},
    "holiday_calendar": {"path": str(HOLIDAY_PATH), "sha256": file_sha256(HOLIDAY_PATH), "rows": len(holidays),
                         "join_key": "measurement_date == holiday date", "nonholiday_default": False},
    "outputs": {"feature_dataset": feature_signature, "canonical_model_dataset": canonical_signature,
                "p95_model_dataset": p95_signature, "p90_model_dataset": p90_signature},
    "population": {"continuous_household_days": len(features), "model_rows": len(canonical),
                   "households": int(canonical["household_id"].nunique()), "p95_positive_events": int(p95_dataset["target"].sum()),
                   "p90_positive_events": int(p90_dataset["target"].sum())},
    "features": {"common": COMMON_FEATURES, "p95_relative": P95_RELATIVE_FEATURES,
                 "p95_event_history": P95_EVENT_FEATURES, "p90_relative": P90_RELATIVE_FEATURES,
                 "p90_event_history": P90_EVENT_FEATURES},
    "missing_value_policy": "preserve feature NaN in CSV; imputation and missing indicators fit only on training data",
    "feature_quality_policy_status": FEATURE_QUALITY_POLICY_STATUS,
    "feature_quality_configuration_id": FEATURE_QUALITY_POLICY_ID,
    "outer_split": "not assigned here; only 04_split_design.ipynb creates the chronological split",
    "validation": {f"{row.stage}_{row.check}": bool(row.passed) for row in all_checks.itertuples(index=False) if pd.notna(row.passed)},
    "software": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
}
write_json(manifest, FEATURE_TABLE_DIR / "04_run_manifest.json")
print("Feature engineering completed; all validation checks passed.")
print(f"All-day features: {FEATURE_OUTPUT_PATH} ({len(features):,} rows)")
print(f"Canonical M0: {CANONICAL_OUTPUT_PATH} ({len(canonical):,} rows; no split assigned)")
print(f"P95 primary dataset: {P95_OUTPUT_PATH} ({int(p95_dataset['target'].sum()):,} positive events)")
print(f"P90 sensitivity dataset: {P90_OUTPUT_PATH} ({int(p90_dataset['target'].sum()):,} positive events)")
print(f"Run manifest: {FEATURE_TABLE_DIR / '04_run_manifest.json'}")


### Handoff to the next stage

The `05_model_ready_m0*.csv` files have matching ordered `row_id`, household, and target-date populations, distinct target-specific histories, no weather, and no `split_name`. Next: rerun model-ready EDA on these rebuilt outputs; then assign one common chronological split in `04_split_design.ipynb`. Missing predictors should be imputed only inside training-fitted model pipelines.
